In [1]:
import tensorflow as tf

In [2]:
print(tf.__version__)

2.10.0


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import librosa
import os

## Data Preprocessing (Converting to Mel Spectrograms)

In [4]:
files = [os.path.join(r, f) for r, _, fs in os.walk('/Volumes/aid_elephants_interaction/Audio Data/2025') for f in fs if ".WAV" in f]

In [5]:
# should be 32220
print(len(files))

32220


In [6]:
import os
import matplotlib
matplotlib.use('Agg') # No pictures displayed 
import pylab
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd
import joblib
import math
from tqdm import tqdm

In [7]:
# where the melspecs will be generated. Pathway needs to exist, next chunk will create class folders IN the folder
mel_spec_dir = "results/melspecs/"

# melspec dir from previous script
# wav_dir = '../data/wavs_2sec/'

# where class metadata lives
# cutoff_class_name = joblib.load('../data/cutoff_class_name_demo.pkl')
# cutoff_class_name

roi_len = 2.000 # IMPORTANT for mfcc duration

In [8]:
# Make a folder for each file where the respective melspecs will go
# adapted from : build a mel spec directory for each class
for i in tqdm(range(len(files))):
    # extract the name of the .wav file in order to name directories correctly
    file_name = files[i].split('/')[5:]
    file_name = '_'.join(file_name)
    #print(files[i])
    #print(file_name)
    #if(os.path.exists(os.path.join(mel_spec_dir, file_name))):
        #print(file_name, "directory already exists. Skipping...")
        #print("")
    #else:
        #os.mkdir(os.path.join(mel_spec_dir, file_name))
    if not (os.path.exists(os.path.join(mel_spec_dir, file_name))):
        os.mkdir(os.path.join(mel_spec_dir, file_name))

#print("Created melspec dirs:")
#print(os.listdir(mel_spec_dir))
print(len(os.listdir(mel_spec_dir)))

  0%|          | 0/32220 [00:00<?, ?it/s]

100%|██████████| 32220/32220 [00:00<00:00, 285790.21it/s]


32220


In [9]:
roi_len = 2.000 # IMPORTANT for mfcc duration
n_sam = int(22050 * roi_len)

In [26]:
en = 0
roi_len = 2.000 # IMPORTANT for mfcc duration
n_sam = int(22050 * roi_len)


# generate mel specs for each concatenated wav
#for wav_nm in os.listdir(wav_dir):
for wav_nm in tqdm(files):
    # print(wav_nm)
    cnt = 0
    en+=1
    # print('>>',en,'/20')

    # y, sr = librosa.load(os.path.join(wav_dir, wav_nm), sr = 22050)
    y, sr = librosa.load(wav_nm, sr = 22050)
    tot_sam = int(np.shape(y)[0]/n_sam)

    for n in range(tot_sam):

        y_21k = y[n*n_sam:(n+1)*n_sam]

        fig = plt.figure(1, frameon=False)
        fig.set_size_inches(6, 6)
        ax = plt.Axes(fig, [0., 0., 1., 1.])
        ax.set_axis_off()
        fig.add_axes(ax)

        S = librosa.feature.melspectrogram(y=y_21k, 
                                           sr=22050, 
                                           n_mels=128,
                                           fmin = 0,                                     
                                           fmax=11025, 
                                           n_fft=728, 
                                           hop_length=32, 
                                           #win_length = None, 
                                           htk = True)

        librosa.display.specshow(librosa.power_to_db(S ** 1, ref=np.max), fmin=0, y_axis='linear')# , cmap = 'gray')

        #directory = os.path.join(mel_spec_dir, wav_nm.split('.')[0] +'/' + (wav_nm.split('.')[0] + '_' +str(cnt) +'.png'))  # 'test.png'
        #fig.savefig(directory)
        file_name = wav_nm.split('/')[5:]
        file_name = '_'.join(file_name)
        directory = os.path.join(mel_spec_dir, file_name, str(n)+'.png')
        fig.savefig(directory)
        print(directory)
        
        fig.clear()
        ax.cla()
        plt.clf()
        plt.close('all')

        print('Number of ROIs converted here:', cnt)

  0%|          | 0/32220 [00:00<?, ?it/s]

results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/0.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/1.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/2.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/3.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/4.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/5.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/6.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/7.png
Number of ROIs converted here: 0
results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/8.png
Number 

  0%|          | 0/32220 [00:27<?, ?it/s]

results/melspecs/05_FR53_02-11-25 to 02-25-25_20250211_20250211_180000.WAV/119.png
Number of ROIs converted here: 0


KeyboardInterrupt: 

## Inference

In [10]:
model_path = "CQuinn8/CQuinn8-ABGQI-CNN-93420d1/ABGQI-CNN"
model = tf.keras.models.load_model(model_path)

In [11]:
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.40_224 (Funct  (None, 7, 7, 1792)       4363712   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1792)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dense (Dense)               (None, 5)                 8965      
                                                                 
Total params: 4,372,677
Trainable params: 8,965
Non-trainable params: 4,363,712
_________________________________________________________________


In [27]:
# dimensions of our images    
img_width, img_height, img_depth = 224, 224, 3 # 224 pixels x 224 pixels x 3 bands (RGB)
# number of images to predict at once 
batch_size = 50

In [28]:
# function to read in a file path
def process_path(file_path, IMG_HEIGHT, IMG_WIDTH):
    # load the raw data from the file as a string
    img = tf.io.read_file(file_path)
    img = decode_img(img, IMG_HEIGHT, IMG_WIDTH)
    return img #, label

# function that interpretes image after being provided a path in process_path
def decode_img(img, IMG_HEIGHT, IMG_WIDTH):
  # convert the compressed string to a 3D uint8 tensor
  img = tf.image.decode_jpeg(img, channels=3)
  # Use `convert_image_dtype` to convert to floats in the [0,1] range.
  img = tf.image.convert_image_dtype(img, tf.float32)
  # resize the image to the desired size.
  return tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

In [29]:
import os, glob
import pandas as pd

## Predict

In [20]:
#output directory
out_dir = '../results/'

# trained models
# wd ='../../cnn_training/results/'
# checkpoint_path = os.path.join(wd, 'ABGQI-CNN') # IMGNET + S2L CVFOLDS

# data dir for images to predict class
data_path = 'results/melspecs/'

# number of cores
cpus = 2

# print(checkpoint_path)
print(out_dir)


png_dirs = []
for filename in os.listdir(data_path):
    if os.path.isdir(os.path.join(data_path, filename)):
        png_dirs.append(filename)
if len(png_dirs) == 0: 
    png_dirs.append(data_path)

print(len(png_dirs))

../results/
32220


In [37]:
# iterate through each melspec in file directory
# mel_spec_dir = "results/melspecs/file0"
# pngs = glob.glob(os.path.join(mel_spec_dir, "*.png"))  
# print(f"Found {len(pngs)} PNGs in {mel_spec_dir}")

# # output directory for predictions
# out_dir = "results/"
# png_dirs = 
# sigmoid_pred_lst = []

#SEE HOW TO INCORPORATE TQDM
for i, f in (enumerate(tqdm(png_dirs))):
    #temp_dir = f
    temp_dir = os.path.join(data_path, f)
    #print(temp_dir)
    pngs = glob.glob(temp_dir + "*.png")
    #print(pngs)
    #print(temp_dir)
    #temp_class = temp_dir.split('/')[-2] # png parent dir
    #print()
    #print(out_dir)
    if len(pngs) != 0:
        print("Number of pngs", len(pngs))

    # check if the parent output dir exits
    # if(os.path.exists(out_dir)):
    #     print("directory already exists. Moving to predictions...")
    # else:
    #     os.mkdir(out_dir)
    #     print("Created :", out_dir)

    # list to hold each dir of predictions (either all pngs in class of mfccs in wav)
    sigmoid_pred_lst = []
    pred_lst = []

    # file path for class pkl
    # out_path_temp = os.path.join(os.path.join(out_dir, "sigmoid", temp_class))
    # print(out_path_temp)

    # iterate through each melspec in file directory
    # for j in range(len(pngs)):
    #     temp_png = pngs[i] # png

    #     # check if pkl (prediction) for file exists
    #     # if os.path.isfile(out_path_temp):
    #     #     pass # skip file


    #     img_ = process_path(temp_png, IMG_HEIGHT = img_height, IMG_WIDTH = img_width)
    #     img_ = tf.reshape(img_, shape=(1, img_height, img_width, img_depth))

#         # get predictions
#         pred = model.predict(img_, verbose=0, steps=1, callbacks=None, max_queue_size=10,
#                              workers=cpus, use_multiprocessing=False)
        
#         sigmoid_pred = tf.math.sigmoid(pred).numpy()
#         sigmoid_pred_lst.append(sigmoid_pred)
#         pred_lst.append(pred)

# # save label preds
# flat_sigmoid = [item for sublist in sigmoid_pred_lst for item in sublist]
# df_sigmoid = pd.DataFrame(flat_sigmoid)
# df_sigmoid.columns = ["Anthrophony", "Biophony", "Geophony", "Other", "Interference"]

# flat_pred = [item for sublist in pred_lst for item in sublist]
# df_pred = pd.DataFrame(flat_pred)
# df_pred.columns = ["Anthrophony", "Biophony", "Geophony", "Other", "Interference"]

# # write directory csv 
# out_path = os.path.join(out_dir, "melspec_predictions.csv")
# df_sigmoid.to_csv(out_path, index=False)
# print(f"Saved predictions to {out_path}")
# df_pred.to_csv(os.path.join(out_dir, 'Non Sigmoid ' + temp_class + '.csv'))
# df_sigmoid.to_csv(os.path.join(out_dir, temp_class + '.csv'), index = False)

# print('Saved this pkl at:',os.path.join(out_dir, temp_class), "\nLength was:",len(sigmoid_pred_lst))

100%|██████████| 32220/32220 [14:33<00:00, 36.89it/s]   
